In [0]:
%sql DESCRIBE EXTERNAL LOCATION ucext;

In [0]:
%sql
--  DROP SCHEMA IF EXISTS man_catlog.caresync_schema_bronze CASCADE;
-- DROP SCHEMA IF EXISTS man_catlog.caresync_schema_gold CASCADE;
-- DROP SCHEMA IF EXISTS man_catlog.caresync_schema_landing CASCADE;
-- DROP SCHEMA IF EXISTS man_catlog.caresync_schema_silver CASCADE;
-- DROP SCHEMA IF EXISTS man_catlog.caresync_schema_bronze CASCADE;


In [0]:
# dbutils.fs.ls("/Volumes/man_catlog/landing/landing_volume/neon_postgres/doctors/2026/11/24")

In [0]:
# dbutils.fs.ls("/Volumes/man_catlog/landing/landing_volume/neon_postgres/hospitals/2026/11/23/")

In [0]:
import logging, sys, time

# Log level widget: DEBUG shows condition traces, INFO shows only step logs
dbutils.widgets.dropdown('log_level', 'DEBUG', ['DEBUG', 'INFO', 'WARNING'])
log_level = dbutils.widgets.get('log_level').strip().upper()

logger = logging.getLogger("bronze_to_silver")
logger.setLevel(getattr(logging, log_level, logging.INFO))
logger.propagate = False
if not logger.handlers:  # avoid duplicate handlers on re-run
    _h = logging.StreamHandler(sys.stdout)
    _h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%Y-%m-%d %H:%M:%S"))
    logger.addHandler(_h)

_pipeline_start = time.time()

def log_step(step_no, msg, start=None):
    """Log a step message; if start time given, append elapsed seconds."""
    elapsed = f" (took {time.time() - start:.2f}s)" if start else ""
    logger.info(f"[STEP {step_no}] {msg}{elapsed}")

def log_cond(step_no, condition, result, detail=""):
    """Debug-trace a condition check and return its result, so it can wrap an if-condition."""
    logger.debug(f"[STEP {step_no}] CHECK '{condition}' -> {result}{' | ' + detail if detail else ''}")
    return result

# Extra Spark actions (counts) used only for debugging run only when DEBUG is enabled
DEBUG_ENABLED = logger.isEnabledFor(logging.DEBUG)

logger.info(f"===== bronze_to_silver notebook started (log_level={log_level}) =====")

In [0]:

dbutils.widgets.text('table_name','')
dbutils.widgets.text('landing_timestamp','')
dbutils.widgets.text('merge_keys','')
dbutils.widgets.text('load_type','')
# dbutils.widgets.text('year','')
# dbutils.widgets.text('month','')
# dbutils.widgets.text('date','')


table_name = dbutils.widgets.get('table_name')
landing_timestamp = dbutils.widgets.get('landing_timestamp')
merge_keys = dbutils.widgets.get('merge_keys')
load_type = dbutils.widgets.get('load_type')
log_step(1, f"Parameters -> table_name='{table_name}', landing_timestamp='{landing_timestamp}', merge_keys='{merge_keys}', load_type='{load_type}'")
logger.debug(f"[STEP 1] Raw values (repr) -> table_name={table_name!r}, landing_timestamp={landing_timestamp!r}, "
             f"merge_keys={merge_keys!r}, load_type={load_type!r}")
if log_cond(1, "table_name is empty", not table_name):
    logger.error("table_name widget is empty")
    raise ValueError("table_name widget is required")
if log_cond(1, "landing_timestamp is empty", not landing_timestamp):
    logger.warning("[STEP 1] landing_timestamp is empty - bronze filter will likely match 0 rows")
# landing_timestamp = dbutils.widgets.get('year') + '/' + dbutils.widgets.get('month') + '/' + dbutils.widgets.get('date')+'/'
# file_path = '/Volumes/man_catlog/landing/landing_volume/' + source_system + '/' + table_name + '/' + landing_timestamp

In [0]:
print(merge_keys)

In [0]:
_t = time.time()
source_table = f"man_catlog.caresync_schema_bronze.{table_name}"
log_step(2, f"Reading bronze table {source_table} for landing_timestamp='{landing_timestamp}'")
from pyspark.sql import functions as F
logger.debug(f"[STEP 2] Filter: landing_timestamp == {landing_timestamp!r}")
try:
    bronze_df = spark.read.table(source_table).filter(F.col("landing_timestamp") == landing_timestamp)
    bronze_count = bronze_df.count()
    log_step(2, f"Bronze read complete: {bronze_count} rows", _t)
    if log_cond(2, "bronze_count == 0", bronze_count == 0, f"bronze_count={bronze_count}"):
        logger.warning(f"No bronze rows found for landing_timestamp='{landing_timestamp}'")
        if DEBUG_ENABLED:
            # Show which landing_timestamp values actually exist, to spot format mismatches
            _avail = [r[0] for r in spark.read.table(source_table).select("landing_timestamp")
                      .distinct().orderBy(F.col("landing_timestamp").desc()).limit(10).collect()]
            logger.debug(f"[STEP 2] Latest landing_timestamp values in bronze: {_avail}")
except Exception as e:
    logger.exception(f"[STEP 2] Failed reading {source_table}: {e}")
    raise

In [0]:
from pyspark.sql import functions as f
_t = time.time()
log_step(3, "Adding audit columns current_ts, landing_ts")
bronze_updt_df = bronze_df.withColumn('current_ts',f.current_timestamp())\
                     .withColumn('landing_ts',f.current_timestamp())
log_step(3, f"Audit columns added. Columns: {bronze_updt_df.columns}", _t)
bronze_updt_df.show(5)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window

target_table = f"man_catlog.caresync_schema_silver.{table_name}"
load_type_norm = load_type.strip().upper()
_t = time.time()
log_step(4, f"Writing to silver table {target_table} with load_type='{load_type_norm or 'OVERWRITE'}'")
logger.debug(f"[STEP 4] load_type raw={load_type!r} -> normalized={load_type_norm!r}")

_target_exists = spark.catalog.tableExists(target_table)
_rows_before = spark.read.table(target_table).count() if (DEBUG_ENABLED and _target_exists) else None
logger.debug(f"[STEP 4] Target exists={_target_exists}, rows before write={_rows_before}")

if log_cond(4, "load_type in (FULL, OVERWRITE, '')", load_type_norm in ("FULL", "OVERWRITE", "")):
    # Full load: replace target data
    logger.info("[STEP 4] Performing OVERWRITE")
    bronze_updt_df.write.mode("overwrite").option("overwriteSchema", "true") \
                  .saveAsTable(target_table)

elif log_cond(4, "load_type == APPEND", load_type_norm == "APPEND"):
    # Append: add incoming rows to target as-is
    logger.info("[STEP 4] Performing APPEND")
    bronze_updt_df.write.mode("append").saveAsTable(target_table)

elif log_cond(4, "load_type == MERGE", load_type_norm == "MERGE"):
    # Merge (upsert): update existing rows, insert new ones
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    logger.info(f"[STEP 4] Performing MERGE on keys {keys}")
    logger.debug(f"[STEP 4] merge_keys raw={merge_keys!r} -> parsed keys={keys}")
    if log_cond(4, "merge keys list is empty", not keys):
        logger.error("[STEP 4] merge_keys is empty for MERGE load")
        raise ValueError("merge_key widget is required for MERGE load (comma-separated for composite keys)")

    _missing_keys = [k for k in keys if k not in bronze_updt_df.columns]
    log_cond(4, "merge keys missing from source columns", bool(_missing_keys), f"missing={_missing_keys}")

    if log_cond(4, "target table does NOT exist", not _target_exists, target_table):
        # First run: target doesn't exist yet, create it
        logger.info(f"[STEP 4] Target {target_table} not found - creating it")
        bronze_updt_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        # Keep one row per key (latest by current_ts) to avoid multiple-match errors
        w = Window.partitionBy(*keys).orderBy(f.col("current_ts").desc())
        source_df = bronze_updt_df.withColumn("_rn", f.row_number().over(w)) \
                                  .filter("_rn = 1").drop("_rn")
        if DEBUG_ENABLED:
            _src_cnt, _dedup_cnt = bronze_updt_df.count(), source_df.count()
            logger.debug(f"[STEP 4] Dedup by {keys}: {_src_cnt} -> {_dedup_cnt} rows "
                         f"({_src_cnt - _dedup_cnt} duplicates dropped)")

        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        logger.debug(f"[STEP 4] Merge condition: {merge_condition}")

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

        # Log merge metrics from the latest Delta history entry
        _m = DeltaTable.forName(spark, target_table).history(1).select("operationMetrics").first()[0]
        logger.info(f"[STEP 4] MERGE metrics -> source_rows={_m.get('numSourceRows')}, "
                    f"inserted={_m.get('numTargetRowsInserted')}, updated={_m.get('numTargetRowsUpdated')}")
        logger.debug(f"[STEP 4] Full MERGE metrics: {dict(_m)}")

else:
    logger.error(f"[STEP 4] Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Use OVERWRITE, APPEND or MERGE.")

if DEBUG_ENABLED:
    _rows_after = spark.read.table(target_table).count()
    logger.debug(f"[STEP 4] Rows after write={_rows_after} (before={_rows_before})")

log_step(4, f"{load_type_norm or 'OVERWRITE'} load completed for {target_table}", _t)

In [0]:
print(table_name)

In [0]:
%sql
select * from IDENTIFIER('man_catlog.caresync_schema_silver.' || :table_name) limit 10

In [0]:
record_count = bronze_updt_df.count()
log_step(5, f"Record count processed for {table_name}: {record_count}")
logger.info(f"===== bronze_to_silver finished in {time.time() - _pipeline_start:.2f}s =====")
logger.debug(f"[STEP 5] Exiting notebook with value {str(record_count)!r}")
dbutils.notebook.exit(str(record_count))